In [1]:
import json
import pycountry

In [2]:
import requests
from collections import defaultdict

BASE_URL = "https://api.openalex.org"


def search_works(query,country_filter=None,pages=2,per_page=200):
    works = []

    for page in range(1, pages + 1):

        params = {
            "search": query,
            "page": page,
            "per-page": per_page
        }

        if country_filter:
            params[
                "filter"
            ] = f"authorships.institutions.country_code:{country_filter}"

        response = requests.get(
            f"{BASE_URL}/works",
            params=params,
            timeout=30
        )

        if response.status_code != 200:
            print(
                f"Failed page {page}:",
                response.status_code
            )
            continue

        data = response.json()

        works.extend(
            data.get("results", [])
        )

    return works


def extract_authors(works):
    """
    Extract authors from works.
    """

    authors = {}

    for work in works:
        for authorship in work.get("authorships", []):
            author = authorship.get("author", {})

            author_id = author.get("id")

            if not author_id:
                continue

            institutions = authorship.get("institutions", [])

            country_code = None
            institution_name = None

            if institutions:
                country_code = institutions[0].get(
                    "country_code"
                )

                institution_name = institutions[0].get(
                    "display_name"
                )

            if author_id not in authors:

                authors[author_id] = {
                    "author_id": author_id,
                    "name": author.get(
                        "display_name"
                    ),
                    "country_code": country_code,
                    "institution": institution_name
                }

    return list(authors.values())

In [3]:
MIN_CANDIDATES = 100
with open("../data/student.json", "r", encoding="utf-8") as f:
    students = json.load(f)

student = students[0]
print("Student:", student["name"])

country_codes = []

for country_name in student["target_countries"]:

    country = pycountry.countries.get(name=country_name)

    if country:
        country_codes.append(country.alpha_2)

country_filter = "|".join(country_codes)

print("Country Filter:", country_filter)

Student: Priya Sharma
Country Filter: CA


In [5]:
"""
PhD Shortlist Builder - v2
===========================
Search WORKS → Extract AUTHORS → Filter by Country (with fallback)
"""

import json
import time
import requests
import pycountry

BASE_URL = "https://api.openalex.org"
MIN_CANDIDATES = 100


# ─────────────────────────────────────────────
# SEARCH WORKS
# ─────────────────────────────────────────────

def search_works(query, country_filter=None, pages=2, per_page=100):
    """Search OpenAlex works (papers) by query and optional country filter."""
    works = []

    for page in range(1, pages + 1):

        params = {
            "search":   query,
            "page":     page,
            "per-page": per_page
        }

        if country_filter:
            params["filter"] = f"authorships.institutions.country_code:{country_filter}"

        try:
            response = requests.get(
                f"{BASE_URL}/works",
                params=params,
                timeout=30
            )

            if response.status_code != 200:
                print(f"  ⚠️  Failed page {page}: {response.status_code}")
                continue

            data  = response.json()
            batch = data.get("results", [])
            works.extend(batch)

            # Small delay — be polite to API
            time.sleep(0.3)

        except requests.exceptions.RequestException as e:
            print(f"  ⚠️  Request error on page {page}: {e}")
            continue

    return works


# ─────────────────────────────────────────────
# EXTRACT AUTHORS FROM WORKS
# ─────────────────────────────────────────────

def extract_authors(works):
    """
    Extract unique authors from a list of works.
    For each author, tries ALL their institutions (not just [0])
    to find a country_code.
    """
    authors = {}

    for work in works:
        for authorship in work.get("authorships", []):

            author     = authorship.get("author", {})
            author_id  = author.get("id")

            if not author_id:
                continue

            institutions = authorship.get("institutions", [])

            # ── FIX 1: Loop ALL institutions, not just [0] ──
            country_code     = None
            institution_name = None

            for inst in institutions:
                code = inst.get("country_code")
                name = inst.get("display_name")

                if code:                        # found a valid country
                    country_code     = code
                    institution_name = name
                    break                       # stop at first valid one

                if name and not institution_name:
                    institution_name = name     # save name even if no code

            # ── Store author (keep richest data if seen before) ──
            if author_id not in authors:
                authors[author_id] = {
                    "author_id":        author_id,
                    "name":             author.get("display_name"),
                    "country_code":     country_code,
                    "institution":      institution_name,
                    "country_resolved": country_code is not None
                }
            else:
                # Already seen — update if we now have better data
                existing = authors[author_id]

                if not existing["country_code"] and country_code:
                    existing["country_code"]     = country_code
                    existing["institution"]      = institution_name
                    existing["country_resolved"] = True

    return list(authors.values())


# ─────────────────────────────────────────────
# COUNTRY FILTER — WITH FALLBACK FOR NULL
# ─────────────────────────────────────────────

def resolve_country_from_openalex(author_id: str) -> str | None:
    """
    Fallback: call OpenAlex author endpoint directly
    to get the most up-to-date institution + country.
    Used when country_code is null after paper extraction.
    """
    try:
        # Strip base URL prefix to get clean ID
        clean_id = author_id.split("/")[-1]

        response = requests.get(
            f"{BASE_URL}/authors/{clean_id}",
            timeout=15
        )

        if response.status_code != 200:
            return None

        data        = response.json()
        institution = data.get("last_known_institution") or {}
        country     = institution.get("country_code")

        time.sleep(0.2)
        return country

    except requests.exceptions.RequestException:
        return None


def filter_by_country(authors: list, allowed_codes: list) -> list:
    """
    Filter authors to only those in allowed countries.

    3-level strategy per author:
      Level 1 — Use country_code already extracted from papers
      Level 2 — If null: call OpenAlex author API directly
      Level 3 — If still null: SKIP (safer than guessing)
    """
    print(f"\n  Filtering {len(authors)} authors for countries: {allowed_codes}")

    passed                = []
    skipped_wrong_country = 0
    skipped_null_country  = 0
    resolved_via_fallback = 0

    for author in authors:

        country_code = author.get("country_code")

        # ── Level 1: Already have country from paper ──
        if country_code:
            if country_code in allowed_codes:
                passed.append(author)
            else:
                skipped_wrong_country += 1
            continue

        else:
            # ── Level 3: Still null → skip ──
            skipped_null_country += 1
            print(f"      ❌ Could not resolve country — skipping")

    print(f"\n  ── Country Filter Results ──────────────────")
    print(f"  Input authors             : {len(authors)}")
    print(f"  Passed ✅                 : {len(passed)}")
    print(f"  Skipped (wrong country)   : {skipped_wrong_country}")
    print(f"  Skipped (null/unresolved) : {skipped_null_country}")
    print(f"  Resolved via fallback     : {resolved_via_fallback}")

    return passed


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────

def main():
    print("\n" + "="*55)
    print("  PhD SHORTLIST BUILDER v2 — Works-Based Search")
    print("="*55)

    # ── Load student ──
    with open("../data/student.json", "r", encoding="utf-8") as f:
        students = json.load(f)

    student = students[0]
    print(f"\nStudent : {student['name']}")

    # ── Get country codes ──
    country_codes = []
    for country_name in student["target_countries"]:
        country = pycountry.countries.get(name=country_name)
        if country:
            country_codes.append(country.alpha_2)
        else:
            print(f"⚠️  Unknown country: {country_name}")

    country_filter = "|".join(country_codes)
    print(f"Countries : {student['target_countries']}")
    print(f"Codes     : {country_codes}")

    # ════════════════════════════════
    # PASS 1 — Search WITH country filter
    # ════════════════════════════════
    print("\n" + "─"*55)
    print("PASS 1: Searching with country filter")
    print("─"*55)

    unique_authors = {}

    for interest in student["research_interests"]:
        print(f"\n  Searching: '{interest}'")

        works   = search_works(query=interest, country_filter=country_filter, pages=2)
        authors = extract_authors(works)

        print(f"  → {len(works)} papers → {len(authors)} authors extracted")

        for author in authors:
            unique_authors[author["author_id"]] = author

    candidate_authors = list(unique_authors.values())
    print(f"\nPass 1 total unique authors : {len(candidate_authors)}")

    # ── Apply country filter to Pass 1 results ──
    # (API filter is approximate — must verify manually too)
    candidate_authors = filter_by_country(candidate_authors, country_codes)
    print(f"Pass 1 after country filter : {len(candidate_authors)}")

    # ════════════════════════════════
    # PASS 2 — Fallback (no country filter, filter manually)
    # ════════════════════════════════
    if len(candidate_authors) < MIN_CANDIDATES:
        print("\n" + "─"*55)
        print(f"PASS 2: Fallback — only {len(candidate_authors)} found, need {MIN_CANDIDATES}")
        print("─"*55)

        fallback_raw = {}

        for interest in student["research_interests"]:
            print(f"\n  Fallback searching: '{interest}'")

            works   = search_works(query=interest, country_filter=None, pages=5)
            authors = extract_authors(works)

            print(f"  → {len(works)} papers → {len(authors)} authors extracted")

            for author in authors:
                # Only add if NOT already in our list
                if author["author_id"] not in unique_authors:
                    fallback_raw[author["author_id"]] = author

        fallback_list = list(fallback_raw.values())
        print(f"\n  New authors from fallback : {len(fallback_list)}")

        # ── Filter fallback results by country ──
        fallback_filtered = filter_by_country(fallback_list, country_codes)

        # ── Merge with Pass 1 results ──
        for author in fallback_filtered:
            unique_authors[author["author_id"]] = author

        candidate_authors = list(unique_authors.values())

    # ════════════════════════════════
    # FINAL RESULTS
    # ════════════════════════════════
    print("\n" + "="*55)
    print("FINAL RESULTS")
    print("="*55)
    print(f"  Total verified authors : {len(candidate_authors)}")

    print("\n  Sample (first 20):")
    print(f"  {'Name':<30} {'Institution':<35} {'Country'}")
    print(f"  {'─'*30} {'─'*35} {'─'*7}")

    for author in candidate_authors[:20]:
        name    = (author["name"]        or "Unknown")[:29]
        inst    = (author["institution"] or "Unknown")[:34]
        country =  author["country_code"] or "???"
        print(f"  {name:<30} {inst:<35} {country}")

    # ── Save results ──
    with open("../data/filtered_authors.json", "w") as f:
        json.dump(candidate_authors, f, indent=2)

    print(f"\n  💾 Saved to: data/filtered_authors.json")
    return candidate_authors


if __name__ == "__main__":
    main()


  PhD SHORTLIST BUILDER v2 — Works-Based Search

Student : Priya Sharma
Countries : ['Canada']
Codes     : ['CA']

───────────────────────────────────────────────────────
PASS 1: Searching with country filter
───────────────────────────────────────────────────────

  Searching: 'Machine Learning'
  → 200 papers → 1207 authors extracted

  Searching: 'Natural Language Processing'
  → 200 papers → 994 authors extracted

  Searching: 'Computer Vision'
  → 200 papers → 952 authors extracted

Pass 1 total unique authors : 3086

  Filtering 3086 authors for countries: ['CA']
      ❌ Could not resolve country — skipping
      ❌ Could not resolve country — skipping
      ❌ Could not resolve country — skipping
      ❌ Could not resolve country — skipping
      ❌ Could not resolve country — skipping
      ❌ Could not resolve country — skipping
      ❌ Could not resolve country — skipping
      ❌ Could not resolve country — skipping
      ❌ Could not resolve country — skipping
      ❌ Could not 

In [6]:
with open("../data/filtered_authors.json", "r", encoding="utf-8") as f:
        authors = json.load(f)
print(len(authors))
print(authors[0])

1263
{'author_id': 'https://openalex.org/A5062571011', 'name': 'Radford M. Neal', 'country_code': 'CA', 'institution': 'University of Toronto', 'country_resolved': True}


In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed

session = requests.Session()

def get_recent_paper_topics(author_id: str) -> dict:
    """
    Fetch last 5 papers for this author.
    Returns their recent topics AND paper evidence links.
    """
    clean_id = author_id.split("/")[-1]

    url = "https://api.openalex.org/works"
    params = {
        "filter":   f"authorships.author.id:{clean_id}",
        "sort":     "publication_date:desc",  # newest first
        "per_page": 5                          # only last 5 papers
    }

    try:
        response = session.get(url, params=params, timeout=15)
        if response.status_code != 200:
            return {"topics": [], "papers": []}

        works = response.json().get("results", [])

        topics  = []
        papers  = []

        for work in works:

            # ── Extract paper title + link ──
            title     = work.get("title", "")
            paper_url = work.get("doi") or work.get("id", "")
            year      = work.get("publication_year", "")

            if title:
                papers.append({
                    "title": title,
                    "year":  year,
                    "url":   f"https://doi.org/{paper_url}"
                            if paper_url.startswith("10.")
                            else paper_url
                })

            # ── Extract topics from THIS paper ──
            for concept in work.get("concepts", [])[:3]:
                name  = concept.get("display_name", "")
                score = concept.get("score", 0)

                # Only take high confidence topics
                if name and score > 0.3:
                    topics.append(name)

        # Remove duplicate topics
        unique_topics = list(dict.fromkeys(topics))

        return {
            "topics": unique_topics[:8],   # top 8 recent topics
            "papers": papers               # last 5 papers as evidence
        }

    except requests.exceptions.RequestException as e:
        print(f"    ⚠️  Paper fetch error: {e}")
        return {"topics": [], "papers": []}

def enrich_author(author: dict) -> dict:

    author_id = author["author_id"].split("/")[-1]

    url = f"https://api.openalex.org/authors/{author_id}"

    try:
        response = session.get(url, timeout=15)

        if response.status_code != 200:
            return author

        data = response.json()

        author["works_count"] = data.get("works_count", 0)

        author["cited_by_count"] = data.get(
            "cited_by_count", 0
        )

        # author["research_topics"] = [
        #     c.get("display_name")
        #     for c in data.get("x_concepts", [])[:5]
        # ]

        recent = get_recent_paper_topics(author["author_id"])
        author["research_topics"] = recent["topics"]
        author["recent_papers"]   = recent["papers"]

    except requests.exceptions.RequestException:
        pass

    return author



def enrich_all_authors(authors: list) -> list:

    print(f"\nEnriching {len(authors)} authors...")

    enriched = []

    with ThreadPoolExecutor(max_workers=40) as executor:

        future_to_author = {
            executor.submit(
                enrich_author,
                author
            ): author
            for author in authors
        }

        completed = 0

        for future in as_completed(future_to_author):

            completed += 1

            try:
                enriched.append(
                    future.result()
                )

            except Exception as e:
                print("Error:", e)

            if completed % 25 == 0:
                print(
                    f"Completed {completed}/{len(authors)}"
                )

    return enriched

In [8]:
# ─────────────────────────────────────────────
# PROFESSOR VERIFICATION
# ─────────────────────────────────────────────

# Known company/industry keywords — not universities
NON_ACADEMIC_KEYWORDS = [
    "google", "microsoft", "amazon", "meta", "apple",
    "deepmind", "openai", "ibm", "samsung", "huawei",
    "research lab", "inc.", "ltd", "corp", "technologies"
]

def is_academic_institution(institution_name: str) -> bool:
    """
    Check if institution is a university, not a company.
    """
    if not institution_name:
        return False

    name_lower = institution_name.lower()

    # ── Reject known companies ──
    for keyword in NON_ACADEMIC_KEYWORDS:
        if keyword in name_lower:
            return False

    # ── Accept known academic words ──
    academic_keywords = [
        "university", "université", "universität",
        "college", "institute", "school of",
        "faculty", "polytechnic", "academia"
    ]
    for keyword in academic_keywords:
        if keyword in name_lower:
            return True

    # ── Unknown — give benefit of doubt ──
    return True


def check_signal_1(author: dict) -> bool:
    """
    Signal 1: Has enough papers?
    PhD student typically has < 5 papers.
    Professor typically has 10+.
    """
    works_count = author.get("works_count", 0)
    return works_count >= 10


def check_signal_2(author: dict) -> bool:
    """
    Signal 2: Has enough citations?
    PhD student: 0–50 citations
    Professor: 500+ citations
    We use 100 as a safe lower threshold.
    """
    cited_by_count = author.get("cited_by_count", 0)
    return cited_by_count >= 200


def check_signal_3(author: dict) -> bool:
    """
    Signal 3: Is institution a real university?
    Rejects companies like Google, Microsoft etc.
    """
    institution = author.get("institution", "")
    return is_academic_institution(institution)


def verify_professor(author: dict) -> dict:
    """
    Run all 3 signals and compute a score.
    Author needs score >= 3 to be considered a real PI.

    Returns author with added fields:
      - pi_score       : 0, 1, 2, or 3
      - is_pi          : True / False
      - pi_fail_reason : why they were rejected (if any)
    """
    s1 = check_signal_1(author)   # enough papers?
    s2 = check_signal_2(author)   # enough citations?
    s3 = check_signal_3(author)   # academic institution?

    score = sum([s1, s2, s3])
    is_pi = score >= 3

    # Build fail reason for transparency
    fail_reasons = []
    if not s1:
        fail_reasons.append(
            f"only {author.get('works_count', 0)} papers (need 10+)"
        )
    if not s2:
        fail_reasons.append(
            f"only {author.get('cited_by_count', 0)} citations (need 200+)"
        )
    if not s3:
        fail_reasons.append(
            f"institution '{author.get('institution')}' not academic"
        )

    author["pi_score"]       = score
    author["is_pi"]          = is_pi
    author["pi_fail_reason"] = ", ".join(fail_reasons) if fail_reasons else None

    return author


def filter_professors(authors: list) -> list:
    """
    Run PI verification on all authors.
    Returns only verified professors.
    """
    print(f"\n{'='*50}")
    print("PROFESSOR VERIFICATION")
    print(f"{'='*50}")
    print(f"  Checking {len(authors)} authors...\n")

    verified    = []
    rejected    = []

    for author in authors:
        result = verify_professor(author)

        if result["is_pi"]:
            verified.append(result)
        else:
            rejected.append(result)
            print(
                f"  SKIP  {author['name']:<30}"
                f" | score={result['pi_score']}"
                f" | {result['pi_fail_reason']}"
            )

    print(f"\n  ── Verification Results ──")
    print(f"  Input      : {len(authors)}")
    print(f"  Verified PI: {len(verified)}")
    print(f"  Rejected   : {len(rejected)}")

    return verified

In [9]:
enriched_authors = enrich_all_authors(authors)    


Enriching 1263 authors...
Completed 25/1263
Completed 50/1263
Completed 75/1263
Completed 100/1263
Completed 125/1263
Completed 150/1263
Completed 175/1263
Completed 200/1263
Completed 225/1263
Completed 250/1263
Completed 275/1263
Completed 300/1263
Completed 325/1263
Completed 350/1263
Completed 375/1263
Completed 400/1263
Completed 425/1263
Completed 450/1263
Completed 475/1263
Completed 500/1263
Completed 525/1263
Completed 550/1263
Completed 575/1263
Completed 600/1263
Completed 625/1263
Completed 650/1263
Completed 675/1263
Completed 700/1263
Completed 725/1263
Completed 750/1263
Completed 775/1263
Completed 800/1263
Completed 825/1263
Completed 850/1263
Completed 875/1263
Completed 900/1263
Completed 925/1263
Completed 950/1263
Completed 975/1263
Completed 1000/1263
Completed 1025/1263
Completed 1050/1263
Completed 1075/1263
Completed 1100/1263
Completed 1125/1263
Completed 1150/1263
Completed 1175/1263
Completed 1200/1263
Completed 1225/1263
Completed 1250/1263


In [10]:
enriched_authors[0]

{'author_id': 'https://openalex.org/A5013283012',
 'name': 'Fangxiang Feng',
 'country_code': 'CA',
 'institution': 'Université de Montréal',
 'country_resolved': True,
 'works_count': 77,
 'cited_by_count': 3972,
 'research_topics': ['Paraphrase',
  'Phrase',
  'Computer science',
  'Artificial intelligence',
  'Schema (genetic algorithms)',
  'Leverage (statistics)',
  'Closed captioning',
  'Inference'],
 'recent_papers': [{'title': 'A fine-grained entity understanding network for weakly supervised phrase grounding',
   'year': 2026,
   'url': 'https://doi.org/10.1016/j.patcog.2026.114075'},
  {'title': 'Data-Bridge: A Multi-Agent System for Code-Based Multimodal Schema Alignment',
   'year': 2026,
   'url': 'https://doi.org/10.1109/icassp55912.2026.11464693'},
  {'title': 'Diffusion-Assisted Progressive Learning for Weakly Supervised Phrase Localization',
   'year': 2026,
   'url': 'https://doi.org/10.1609/aaai.v40i38.40473'},
  {'title': 'Towards balancing the efficiency and effec

In [11]:
verified_professors = filter_professors(enriched_authors)


PROFESSOR VERIFICATION
  Checking 1263 authors...

  SKIP  Nathan Killoran                | score=2 | institution 'Xanadu Quantum Technologies (Canada)' not academic
  SKIP  Maxim Milakov                  | score=2 | only 6 papers (need 10+)
  SKIP  Dimitris Athanasakis           | score=2 | only 6 papers (need 10+)
  SKIP  Maria Schuld                   | score=2 | institution 'Xanadu Quantum Technologies (Canada)' not academic
  SKIP  Will Cukierski                 | score=2 | only 7 papers (need 10+)
  SKIP  Romeo Chalil                   | score=2 | only 1 papers (need 10+)
  SKIP  Keaton W Smith                 | score=2 | only 6 papers (need 10+)
  SKIP  Antoine Prouvost               | score=2 | only 9 papers (need 10+)
  SKIP  Sohaib A Syed                  | score=2 | only 1 papers (need 10+)
  SKIP  Mateusz A. Wlodarski           | score=2 | only 4 papers (need 10+)
  SKIP  Mugdha Dave                    | score=2 | only 4 papers (need 10+)
  SKIP  Komal Kaur                

In [12]:
verified_professors[0]

{'author_id': 'https://openalex.org/A5013283012',
 'name': 'Fangxiang Feng',
 'country_code': 'CA',
 'institution': 'Université de Montréal',
 'country_resolved': True,
 'works_count': 77,
 'cited_by_count': 3972,
 'research_topics': ['Paraphrase',
  'Phrase',
  'Computer science',
  'Artificial intelligence',
  'Schema (genetic algorithms)',
  'Leverage (statistics)',
  'Closed captioning',
  'Inference'],
 'recent_papers': [{'title': 'A fine-grained entity understanding network for weakly supervised phrase grounding',
   'year': 2026,
   'url': 'https://doi.org/10.1016/j.patcog.2026.114075'},
  {'title': 'Data-Bridge: A Multi-Agent System for Code-Based Multimodal Schema Alignment',
   'year': 2026,
   'url': 'https://doi.org/10.1109/icassp55912.2026.11464693'},
  {'title': 'Diffusion-Assisted Progressive Learning for Weakly Supervised Phrase Localization',
   'year': 2026,
   'url': 'https://doi.org/10.1609/aaai.v40i38.40473'},
  {'title': 'Towards balancing the efficiency and effec

In [13]:
# Score matching
with open("../data/student.json","r",encoding="utf-8") as f:
    student = json.load(f)[0]

student_text = " ".join(student["research_interests"])
student_text += " "
student_text += " ".join(student["skills"])

print(student_text)

Machine Learning Natural Language Processing Computer Vision Python PyTorch Transformers


In [14]:
def build_author_text(author):

    paper_titles = []

    for paper in author.get("recent_papers",[]):
        if paper.get("title"):
            paper_titles.append(paper["title"])

    return ";".join(paper_titles)

In [15]:
build_author_text(verified_professors[0])

'A fine-grained entity understanding network for weakly supervised phrase grounding;Data-Bridge: A Multi-Agent System for Code-Based Multimodal Schema Alignment;Diffusion-Assisted Progressive Learning for Weakly Supervised Phrase Localization;Towards balancing the efficiency and effectiveness: a unified edit-based framework for automatic image captioning;Diffusion-Assisted Progressive Learning for Weakly Supervised Phrase Localization'

In [16]:
from sentence_transformers import (SentenceTransformer,util)

model = SentenceTransformer("all-MiniLM-L6-v2")

student_embedding = model.encode(student_text,convert_to_tensor=True
)

/home/dibyendu/pythonProject/phd_shortlist_builder/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2070.71it/s]


In [17]:
def calculate_similarity(author):

    author_text = build_author_text(author)

    author_embedding = model.encode(author_text,convert_to_tensor=True)

    similarity = util.cos_sim(
        student_embedding,
        author_embedding
    ).item()

    return similarity

In [18]:
for author in verified_professors:

    similarity = calculate_similarity(
        author
    )

    author["similarity_score"] = similarity

In [19]:
verified_professors.sort(
    key=lambda x:
        x["similarity_score"],
    reverse=True
)

top_60 = verified_professors[:60]

In [20]:
for i in range(5):
    print(
        i,
        top_60[i]["name"],
        top_60[i]["similarity_score"]
    )

0 Sagi Eppel 0.42428576946258545
1 Ben Hamner 0.40728098154067993
2 Naveed Ejaz 0.39516329765319824
3 Greg Mori 0.39047661423683167
4 William McNally 0.38151633739471436


In [21]:
top_60[0]

{'author_id': 'https://openalex.org/A5021796022',
 'name': 'Sagi Eppel',
 'country_code': 'CA',
 'institution': 'University of Toronto',
 'country_resolved': True,
 'works_count': 67,
 'cited_by_count': 481,
 'research_topics': ['Artificial intelligence',
  'Computer science',
  'Texture (cosmology)',
  'Computer vision',
  'Pattern recognition (psychology)',
  'Shot (pellet)',
  'Segmentation',
  'Natural (archaeology)'],
 'recent_papers': [{'title': 'Shape and Texture Recognition in Large Vision-Language Models',
   'year': 2025,
   'url': 'https://doi.org/10.48550/arxiv.2503.23062'},
  {'title': 'Do large language vision models understand 3D shapes?',
   'year': 2024,
   'url': 'https://doi.org/10.48550/arxiv.2412.10908'},
  {'title': 'Vastextures: Vast repository of textures and PBR materials extracted from real-world images using unsupervised methods',
   'year': 2024,
   'url': 'https://doi.org/10.48550/arxiv.2406.17146'},
  {'title': 'Learning Zero-Shot Material States Segmentat

In [22]:
import os
import json
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import PromptTemplate
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def assign_tier(similarity_score: float) -> str:
    """
    Assign tier based on cosine similarity score.
 
    Thresholds chosen based on cosine similarity distribution:
      High similarity  (0.45+) → reach   (top match, competitive PI)
      Medium           (0.30+) → target  (good realistic match)
      Lower            (<0.30) → safety  (weaker match but still relevant)
    """
    if similarity_score >= 0.45:
        return "reach"
    elif similarity_score >= 0.30:
        return "target"
    else:
        return "safety"
    
# ─────────────────────────────────────────────
# BATCH WHY_MATCH GENERATION (6 calls instead of 60)
# ─────────────────────────────────────────────

BATCH_PROMPT = PromptTemplate(
    input_variables=[
        "student_name",
        "student_interests", 
        "student_background",
        "professors_batch"
    ],
    template="""
You are helping a student write personalised PhD application emails.

STUDENT NAME: {student_name}
STUDENT RESEARCH INTERESTS: {student_interests}
STUDENT BACKGROUND: {student_background}

Below are {batch_size} professors. For EACH professor write a personalised
3-4 sentence why_match explaining why they are a good supervisor match.

Rules:
- Be SPECIFIC — mention professor's actual research topics
- Connect it to student's actual interests
- NO generic phrases like "highly regarded" or "excellent mentor"

PROFESSORS:
{professors_batch}

Reply ONLY in this exact JSON format, nothing else:
[
  {{
    "author_id": "https://openalex.org/A123",
    "why_match": "2-3 sentences here..."
  }},
  ...
]
"""
)


def format_batch_for_prompt(batch: list) -> str:
    """
    Format a batch of professors into readable text for the prompt.
    """
    lines = []
    for i, prof in enumerate(batch, 1):
        papers = " | ".join([
            p["title"] for p in prof.get("recent_papers", [])[:2]
        ])
        topics = ", ".join(prof.get("research_topics", [])[:4])
        
        lines.append(
            f"{i}. ID: {prof['author_id']}\n"
            f"   Name: {prof['name']}\n"
            f"   Institution: {prof['institution']}\n"
            f"   Topics: {topics}\n"
            f"   Recent Papers: {papers}"
        )
    return "\n\n".join(lines)


def generate_why_match_batch(
    batch: list,
    student: dict
) -> dict:
    """
    Generate why_match for a batch of 10 professors in ONE LLM call.
    Returns dict: { author_id → why_match string }
    """
    professors_text = format_batch_for_prompt(batch)

    prompt_text = BATCH_PROMPT.format(
        student_name       = student.get("name", "the student"),
        student_interests  = ", ".join(student.get("research_interests", [])),
        student_background = student.get("intro_call_summary", ""),
        professors_batch   = professors_text,
        batch_size         = len(batch)
    )

    try:
        response = llm.invoke([HumanMessage(content=prompt_text)])
        raw      = response.content.strip()

        # Strip markdown code fences if LLM adds them
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]

        results  = json.loads(raw)

        # Convert list → dict keyed by author_id
        return {
            item["author_id"]: item["why_match"]
            for item in results
        }

    except json.JSONDecodeError as e:
        print(f"    ⚠️  JSON parse error in batch: {e}")
        # Fallback: return empty so individual fallback kicks in
        return {}

    except Exception as e:
        print(f"    ⚠️  Batch LLM error: {e}")
        return {}


def generate_all_why_matches(
    top_60: list,
    student: dict,
    batch_size: int = 10
) -> dict:
    """
    Process all professors in batches of 10.
    Returns dict: { author_id → why_match string }

    60 professors / 10 per batch = 6 API calls total
    """
    print(f"\n  Generating why_match in batches of {batch_size}...")
    print(f"  Total professors : {len(top_60)}")
    print(f"  Total API calls  : {len(top_60) // batch_size + 1}\n")

    all_why_matches = {}

    # Split into batches
    batches = [
        top_60[i : i + batch_size]
        for i in range(0, len(top_60), batch_size)
    ]

    for i, batch in enumerate(batches, 1):
        print(f"  Batch [{i}/{len(batches)}] — {len(batch)} professors...")

        batch_results = generate_why_match_batch(batch, student)

        if batch_results:
            all_why_matches.update(batch_results)
            print(f"    ✅ Got {len(batch_results)} why_matches")
        else:
            # Batch failed — generate fallback for each professor
            print(f"    ⚠️  Batch failed — using fallback text")
            for prof in batch:
                topics = ", ".join(prof.get("research_topics", [])[:3])
                interests = ", ".join(student.get("research_interests", [])[:2])
                all_why_matches[prof["author_id"]] = (
                    f"Prof. {prof['name']}'s work on {topics} "
                    f"aligns with the student's interest in {interests}."
                )

    print(f"\n  Total why_matches generated: {len(all_why_matches)}")
    return all_why_matches


In [31]:
def generate_shortlist(top_60: list, student: dict, output_dir: str = "../sample_output") -> dict:

    print("\n" + "="*55)
    print("  GENERATING FINAL SHORTLIST")
    print("="*55)

    # ── Generate ALL why_matches in 6 batch calls ──
    all_why_matches = generate_all_why_matches(top_60, student, batch_size=10)

    shortlist = []

    for professor in top_60:
        author_id = professor["author_id"]

        # Get why_match from batch results
        why_match = all_why_matches.get(author_id)

        # If missing for any reason use fallback
        if not why_match:
            topics    = ", ".join(professor.get("research_topics", [])[:3])
            interests = ", ".join(student.get("research_interests", [])[:2])
            why_match = (
                f"Prof. {professor['name']}'s work on {topics} "
                f"aligns with the student's interest in {interests}."
            )

        entry = build_entry(professor, student, why_match)
        shortlist.append(entry)

    # # Sort by similarity score
    # shortlist.sort(key=lambda x: x["similarity_score"], reverse=True)

    output = {
        "student_name": student.get("name"),
        "generated_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "total":        len(shortlist),
        "tier_summary": {
            "reach":  sum(1 for e in shortlist if e["tier"] == "reach"),
            "target": sum(1 for e in shortlist if e["tier"] == "target"),
            "safety": sum(1 for e in shortlist if e["tier"] == "safety"),
        },
        "shortlist": shortlist
    }

    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, "result.json")

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"\n  ✅ Saved → {filepath}")
    print(f"  Total  : {output['total']}")
    print(f"  Reach  : {output['tier_summary']['reach']}")
    print(f"  Target : {output['tier_summary']['target']}")
    print(f"  Safety : {output['tier_summary']['safety']}")

    return output


# Updated build_entry — now accepts why_match as parameter
def build_entry(professor: dict, student: dict, why_match: str) -> dict:
    return {
        "name":           professor.get("name"),
        "institution":    professor.get("institution"),
        "country":        professor.get("country_code"),
        "contact_email":  None,
        "research_focus": ", ".join(professor.get("research_topics", [])[:5]),
        "evidence": {
            "papers": professor.get("recent_papers", []),
            "openalex_profile": professor.get("author_id")
        },
        "why_match":        why_match,
        "tier":             assign_tier(professor["similarity_score"])
    }

In [32]:
with open("../data/student.json", "r", encoding="utf-8") as f:
    student = json.load(f)[0]

result = generate_shortlist(top_60, student)


  GENERATING FINAL SHORTLIST

  Generating why_match in batches of 10...
  Total professors : 60
  Total API calls  : 7

  Batch [1/6] — 10 professors...
    ✅ Got 10 why_matches
  Batch [2/6] — 10 professors...
    ✅ Got 10 why_matches
  Batch [3/6] — 10 professors...
    ✅ Got 10 why_matches
  Batch [4/6] — 10 professors...
    ✅ Got 10 why_matches
  Batch [5/6] — 10 professors...
    ✅ Got 10 why_matches
  Batch [6/6] — 10 professors...
    ✅ Got 10 why_matches

  Total why_matches generated: 60

  ✅ Saved → ../sample_output/result.json
  Total  : 60
  Reach  : 0
  Target : 60
  Safety : 0


In [22]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# =====================================================
# Load Environment Variables
# =====================================================
load_dotenv()

# =====================================================
# Load Student
# =====================================================
with open("../data/student.json", "r", encoding="utf-8") as f:
    student = json.load(f)[0]

# =====================================================
# Pydantic Models
# =====================================================
class ProfessorEvaluation(BaseModel):
    name: str
    match_score: int = Field(description="Match score between 0 and 100")
    research_focus: list[str]
    why_match: str


class ProfessorEvaluationList(BaseModel):
    evaluations: list[ProfessorEvaluation]


# =====================================================
# LLM
# =====================================================
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

structured_llm = llm.with_structured_output(
    ProfessorEvaluationList
)

# =====================================================
# Prompt
# =====================================================
prompt = ChatPromptTemplate.from_template("""
You are an expert PhD admissions and research matching consultant.

Your task is to evaluate each professor independently and determine
how suitable they are as a potential PhD supervisor for the student.

Scoring Criteria:

1. Research Alignment (70%)
   - Overlap between student's interests and professor's research.
   - Relevance of recent publications.

2. Skills Alignment (20%)
   - Whether the student's skills can contribute to the professor's work.

3. Future Research Potential (10%)
   - Likelihood of productive collaboration.

Scoring Guide:

90-100 : Exceptional fit
75-89  : Strong fit
60-74  : Moderate fit
40-59  : Weak fit
0-39   : Poor fit

Student:

Research Interests:
{research_interests}

Skills:
{skills}

Professors:

{professors}

Instructions:

- Evaluate every professor independently.
- Use the full score range.
- Focus primarily on research overlap.
- Mention overlapping topics explicitly.
- Keep explanations concise (2-4 sentences).
- Do not compare professors against each other.
""")

chain = prompt | structured_llm

# =====================================================
# Utility Functions
# =====================================================
def create_batches(items, batch_size=10):
    return [
        items[i:i + batch_size]
        for i in range(0, len(items), batch_size)
    ]


def evaluate_batch(batch):

    professor_text = "\n\n".join(
        [
            f"""
Professor Name: {prof.get('name', '')}
Institution: {prof.get('institution', '')}
Country: {prof.get('country_code', '')}

Research Topics:
{", ".join(prof.get("research_topics", []))}

Recent Papers:
{chr(10).join(
    f"- {paper.get('title', '')} ({paper.get('year', '')})"
    for paper in prof.get("recent_papers", [])
)}
"""
            for prof in batch
        ]
    )

    result = chain.invoke(
        {
            "research_interests": ", ".join(
                student.get("research_interests", [])
            ),
            "skills": ", ".join(
                student.get("skills", [])
            ),
            "professors": professor_text,
        }
    )

    return result.evaluations


# =====================================================
# INPUT PROFESSORS
# =====================================================
# Assumes top_60 already exists in memory
# and contains your professor list

candidate_professors = top_60

# =====================================================
# Run Evaluation
# =====================================================
all_results = []

batches = create_batches(
    candidate_professors,
    batch_size=10
)

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = [
        executor.submit(
            evaluate_batch,
            batch
        )
        for batch in batches
    ]

    for future in as_completed(futures):
        try:
            all_results.extend(
                future.result()
            )
        except Exception as e:
            print(f"Batch failed: {e}")

# =====================================================
# Final Ranking
# =====================================================
all_results.sort(
    key=lambda x: x.match_score,
    reverse=True
)

# =====================================================
# Display Top Matches
# =====================================================
print("\n" + "=" * 80)
print("TOP PROFESSOR MATCHES")
print("=" * 80)

for rank, prof in enumerate(all_results, start=1):

    print(f"\n#{rank}")
    print(f"Professor      : {prof.name}")
    print(f"Match Score    : {prof.match_score}")

    if prof.research_focus:
        print(
            f"Research Focus : {', '.join(prof.research_focus)}"
        )

    print(f"Why Match      : {prof.why_match}")

# =====================================================
# Optional: Save Results
# =====================================================
with open(
    "professor_rankings.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        [
            {
                "rank": idx + 1,
                "name": p.name,
                "match_score": p.match_score,
                "research_focus": p.research_focus,
                "why_match": p.why_match,
            }
            for idx, p in enumerate(all_results)
        ],
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    f"\nSaved {len(all_results)} rankings to professor_rankings.json"
)


TOP PROFESSOR MATCHES

#1
Professor      : Hossein Hajimirsadeghi
Match Score    : 90
Research Focus : Reinforcement learning, Transformer, Artificial intelligence
Why Match      : Professor Hajimirsadeghi's research on transformers and reinforcement learning closely aligns with the student's interests in machine learning and NLP. His recent papers on reasoning and decoding methods suggest a high potential for collaboration, making him an exceptional fit.

#2
Professor      : Esma Aı̈meur
Match Score    : 90
Research Focus : Natural language processing, Artificial intelligence, Fake news detection
Why Match      : The research on fake news detection and NLP is highly relevant to the student's interests. The professor's recent work directly aligns with the student's skills in machine learning and NLP, making this an exceptional fit.

#3
Professor      : Diana Inkpen
Match Score    : 90
Research Focus : Natural language processing, Transformer, Computer science
Why Match      : Professo

In [26]:
from pydantic import BaseModel
from typing import List

class ProfessorEvaluation(BaseModel):
    name: str
    match_score: int
    research_focus: List[str]
    why_match: str
    is_good_match: bool

class ProfessorEvaluationList(BaseModel):
    professors: List[ProfessorEvaluation]

In [27]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

with open("../data/student.json","r",encoding="utf-8") as f:
    student = json.load(f)[0]

prompt = ChatPromptTemplate.from_template("""
You are helping match PhD students
with supervisors.

STUDENT

Research Interests:
{research_interests}

Skills:
{skills}

PROFESSORS

{professors}

For EACH professor return JSON.

Example:

[
  {{
    "name": "...",
    "match_score": 0,
    "research_focus": [],
    "why_match": ""
  }}
]

Return ONLY JSON.
""")

chain = prompt | llm.with_structured_output(ProfessorEvaluationList)

In [28]:
def chunk_list(data, size):
    return [data[i:i + size] for i in range(0, len(data), size)]

def create_professor_block(professors):

    return "\n".join(
        f"""
Professor {i}

Name: {p['name']}
Institution: {p['institution']}
Research Topics: {", ".join(p.get('research_topics', []))}

Recent Papers:
{chr(10).join(x['title'] for x in p.get('recent_papers', []) if x.get('title'))}
"""
        for i, p in enumerate(professors, 1)
    )

def evaluate_batch(batch):

    return chain.invoke({
        "research_interests": ", ".join(student["research_interests"]),
        "skills": ", ".join(student["skills"]),
        "professors": create_professor_block(batch)
    })

In [29]:
results = []

for i, batch in enumerate(chunk_list(top_60, 10), 1):

    print(f"Processing Batch {i}")

    results.extend(
        evaluate_batch(batch).professors
    )

Processing Batch 1
Processing Batch 2
Processing Batch 3
Processing Batch 4
Processing Batch 5
Processing Batch 6


In [30]:
result_map = {r.name: r for r in results}

In [31]:
for prof in top_60:

    if prof["name"] not in result_map:
        continue

    r = result_map[prof["name"]]

    prof["llm_match_score"] = r.match_score
    prof["research_focus"] = r.research_focus
    prof["why_match"] = r.why_match
    prof["is_good_match"] = r.is_good_match

In [33]:
top_60[50]

{'author_id': 'https://openalex.org/A5073426758',
 'name': 'Mahdi S. Hosseini',
 'country_code': 'CA',
 'institution': 'University of New Brunswick',
 'country_resolved': True,
 'works_count': 83,
 'cited_by_count': 883,
 'research_topics': ['Computer science',
  'Artificial intelligence',
  'Contextual image classification',
  'Deep learning',
  'Leverage (statistics)',
  'Digital pathology',
  'Interpretability',
  'Jaccard index'],
 'recent_papers': [{'title': 'PathTTT: Test-Time Training with\xa0Meta-auxiliary Learning for\xa0Pathology Image Classification',
   'year': 2025,
   'url': 'https://doi.org/10.1007/978-3-031-96628-6_3'},
  {'title': 'Deep Learning for Automated Crack Recognition: Insights, Challenges, and Future Directions',
   'year': 2025,
   'url': 'https://doi.org/10.22260/isarc2025/0160'},
  {'title': 'ADPv2: A Hierarchical Histological Tissue Type-Annotated Dataset for Potential Biomarker Discovery of Colorectal Disease',
   'year': 2025,
   'url': 'https://doi.org